---
title: Estimación de series temporales de rasgos biofísicos
subject: Ejercicio
subtitle: Ejercicio que muestra cómo obtener imágenes mensuales de rasgos biofísicos mediante imágenes Sentinel-2
authors:
  - name: Héctor Nieto
    affiliations:
      - Instituto de Ciencias Agrarias, ICA
      - CSIC
    orcid: 0000-0003-4250-6424
    email: hector.nieto@ica.csic.es
  - name: Radoslaw Guzinski
    affiliations:
      - DHI
    orcid: 0000-0003-0044-6806
  - name: Benjamin Mary
    affiliation:
      - Instituto de Ciencias Agrarias
      - CSIC
    orcid: 0000-0003-0815-842X
label: nb-biophysical
license: CC-BY-SA-4.0
keywords: Prospect, 4SAIL, crop yield, Daisy crop model
myst:
  enable_extensions: ["deflist", "attrs_block", "attrs_inline"]
jupytext:
  text_representation:
    extension: .md
    format_name: myst
    format_version: 0.13
    jupytext_version: 1.19.1
kernelspec:
  display_name: Python 3 (ipykernel)
  language: python
  name: python3
---

# Introducción

En este ejercicios vamos a pre-procesar en la nube y descargar imágenes mensuales para los biofísicos obtenidos a partir de imágenes Sentinel-2.

Usaremos para ello el entortno del [Copernicus Data Space Ecosystem (CDSE)](https://dataspace.copernicus.eu/) usando la interfaz [openEO](https://openeo.org/).

Este cuaderno puede ejecutarse en el [Jupyterhub de Copernicus Dataspace](https://jupyterhub.dataspace.copernicus.eu), en cuyo caso no se realizan descargas locales de datos, ya que tanto los datos como el entorno de ejecución están en CDSE y se mantienen en tu cuenta.

:::{warning} Atención
Si estás usando el entorno de CDSE debes seleccionar uno de los kernels con GDAL instalado, p. ej. "Geo science".
:::

::::{note} Nota
Las características de Sentinel-2 son las siguientes
:::{table} Características de la misión Sentinel-2
:label: s2
Plataformas | Rango espectral     | Número de bandas | Resolución espacial | Resolución temporal
:---        | :---                | :---             | :---                | :---            
A, B, C     | Visible, NIR, SWIR  | 10 (13)          | 10 -- 20 m          | 5 -- 10 días
:::
::::

Primero comprobamos que el Sen-ET Toolbox esté instalado (y lo instalamos si es necesario) y luego importamos todos los paquetes necesarios.

In [3]:
try:
    import senet_toolbox
    print("senet_toolbox importado correctamente")
except ModuleNotFoundError:
    print("Falta la librería senet_toolbox, instalando desde Git")
    !pip install senet_toolbox@git+https://github.com/DHI/Sen-ET-OpenEO-toolbox.git

senet_toolbox importado correctamente


In [13]:
from pathlib import Path
from dateutil.relativedelta import relativedelta
from shapely import to_geojson
from shapely.geometry import box
from osgeo import gdal
import rasterio
import openeo
import pandas as pd
from joblib import load
from rasterstats import zonal_stats
from senet_toolbox.workflows import collect_input_data
from senet_toolbox.utils import visualization, date_selector
from senet_toolbox.utils.raster_utils import save_raster
from senet_toolbox.workflows import biophysical_processing
from ipywidgets import interact, interactive, fixed, widgets
from IPython.display import display
import datetime as dt
print("Librerías importadas correctamente, puedes continuar")

Librerías importadas correctamente, puedes continuar


### Seleccionar el Área de Interés
Para mantener los datos organizados y facilitar el procesamiento de series temporales, los datos de entrada y salida se guardan en carpetas de Área de Interés (AOI). Todos los datos dentro de una carpeta AOI tienen la misma extensión y cuadrícula.

En la celda siguiente, selecciona la ubicación donde deseas almacenar los datos y el nombre del AOI. Al ejecutar en el Jupyterhub de CDSE, se recomienda mantenerlo dentro de `./mystorage/301-biophysical`, de lo contrario los datos se borrarán entre sesiones.

Si estás configurando un nuevo AOI, dibuja un polígono en el mapa con la extensión que deseas procesar. Se recomienda seleccionar AOIs de pequeñas (unos pocos kilómetros) para agilizar el procesado y no usar los cŕeditos gratuitos rápidamente

Si estás trabajando con un AOI existente, el mapa mostrará su extensión.

In [5]:
data_dir = "./mystorage/301a-biophysical"
aoi_name = "agramon"
aoi_data_dir = Path(data_dir) / aoi_name

In [6]:
# Dibuja o visualiza la extensión del AOI al configurar uno nuevo
map, bboxs = visualization.select_aoi(aoi_data_dir)
map

Map(center=[38.44, -1.6705], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

## Seleccionar el rango de fechas
En la siguiente celda selecciona el año hidrológico de inicio y de final que te interese procesar

In [7]:
w_years = widgets.IntRangeSlider(
    description="Años",
    tooltip='Selecciona el rango de años hidrológicos a procesar',
    disabled=False,
    min=2015,
    max=dt.datetime.today().year - 1,
    value=(2015, dt.datetime.today().year - 1),
)
display(w_years)

IntRangeSlider(value=(2015, 2025), description='Años', max=2025, min=2015, tooltip='Selecciona el rango de año…

## Conectarse al backend de OpenEO

Las imágenes de Sentinel-2 serán procesadas y descargadas desde la interfaz OpenEO de CDSE. Ejecuta la celda siguiente para autenticarte en OpenEO.

:::{important} Importante
Es posible que debas hacer clic en un enlace de autenticación que aparecerá y seguir las instrucciones.
:::

In [14]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

2026-04-27 08:54:20,285 [INFO] Found OIDC providers: ['CDSE']
2026-04-27 08:54:20,286 [INFO] No OIDC provider given, but only one available: 'CDSE'. Using that one.
2026-04-27 08:54:20,754 [INFO] Using default client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e' from OIDC provider 'CDSE' info.
2026-04-27 08:54:20,755 [INFO] Found refresh token: trying refresh token based authentication.
2026-04-27 08:54:20,755 [INFO] Doing 'refresh_token' token request 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token' with post data fields ['grant_type', 'client_id', 'refresh_token'] (client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')
2026-04-27 08:54:21,062 [INFO] Obtained tokens: ['token_type', 'access_token', 'expires_in', 'id_token', 'refresh_token', 'scope']
2026-04-27 08:54:21,063 [INFO] Storing refresh token for issuer 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE' (client 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')


Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

## Descargar series temporales de Sentinel-2

Descargaremos medias mensuales y máximos anuales del LAI y del Contenido de Clorofila del Dosel, que se obtienen a partir del producto de reflectancia en el Fondo de la Atmósfera (BOA) de Sentinel-2 usando el [procesador BIOPAR de OpenEO](https://openeo.dataspace.copernicus.eu/openeo/1.1/processes/u:3e24e251-2e9a-438f-90a9-d4500e576574/BIOPAR):

Tanto el LAI como el CCC se definen en BIOPAR de la siguiente manera:
- **Índice de Área Foliar (LAI)**: la mitad del área total de los elementos verdes del dosel por unidad de superficie horizontal del suelo. El valor derivado por satélite corresponde al LAI verde total de todas las capas del dosel, incluido el sotobosque, que puede representar una contribución muy significativa, especialmente en bosques.

- **Contenido de Clorofila del Dosel (CCC)**: el contenido total de clorofila por unidad de superficie del suelo en un grupo continuo de plantas. Es muy adecuado para cuantificar el contenido de nitrógeno a nivel del dosel y estimar la producción primaria bruta.

La metodología BIOPAR fue desarrollada inicialmente para generar productos biofísicos a partir de los sensores SPOT-VEGETATION, ENVISAT-MERIS, SPOT-HRVIR y LANDSAT-OLI, y fue posteriormente adaptada para Sentinel-2. Consiste principalmente en simular una base de datos exhaustiva de reflectancias de dosel (BOA) a partir de las características de la vegetación y la geometría de observación e iluminación. A continuación, se entrenan redes neuronales para estimar una serie de estas características del dosel (BIOPARs) a partir de las reflectancias BOA simuladas junto con los ángulos que definen la configuración observacional.

:::{seealso} Ver también
[Weiss y Baret (2016). S2ToolBox Level 2 products: LAI, FAPAR, FCOVER Version 1.1](http://step.esa.int/docs/extra/ATBD_S2ToolBox_L2B_V1.1.pdf)
:::

:::{important} Importante
Para áreas grandes, la descarga y agregación de datos en OpenEO puede tardar bastante y podría fallar. Se recomienda procesar regiones más pequeñas a la vez.
Accede a [https://openeo.dataspace.copernicus.eu/](https://openeo.dataspace.copernicus.eu/) e inicia sesión para hacer seguimiento de los trabajos y ver posibles errores.
:::

### Descargar promedios mensuales de LAI

In [16]:
MAX_JOBS = 24
        
date_ini = dt.datetime(w_years.value[0], 10, 1)
date_end = dt.datetime(w_years.value[1], 9, 30)
bbox = bboxs[0]
bbox_polygon = eval(to_geojson(box(*bbox)))
var = "LAI"
input_dir = aoi_data_dir / "input"

if not input_dir.exists():
    input_dir.mkdir()

time_window = [str(date_ini.date()), str(date_end.date())]
print(f"Procesando BIOPAR {var} desde {time_window[0]} hasta {time_window[1]}\n"
      f"para la extensión {bbox}, esto puede tardar un momento")

bio = biophysical_processing.get_biopar(
    connection, var, time_window, bbox_polygon
    )

date = date_ini
jobs = []
count = 0
while date < date_end:    
    dates_range = [str(date.date()),  
                   str((date + relativedelta(months=1) - dt.timedelta(days=1)).date())]    
    bio_month = bio.filter_temporal(dates_range).reduce_dimension(dimension="t", reducer="mean")  

    s2_path = input_dir / f"s2_{date:%Y%m}_{var}.tif"
    if not s2_path.exists():
        print(f"Creando trabajo para el {var} mensual de Sentinel-2 desde {dates_range[0]} hasta {dates_range[1]}")
        job = bio_month.create_job(out_format="GTiff")
        job.start()
        jobs.append([job, s2_path])
    else:
        print(f"Se encontraron datos de Sentinel-2 en caché para el mes {date:%m} de {date:%Y}. Omitiendo descarga.")  
    
    date += relativedelta(months=1)
    count += 1
    # Download data every MAX_DOWNLOADS
    if count > MAX_JOBS:
        for job, path in jobs:
            print(f"Procesando y descargando en {path}", end="...") 
            collect_input_data.wait_and_download(job, path, poll_interval=60)
            print(f"Descargado") 
        # Restart job list and counter
        count = 0        
        jobs = []

# Process the last batch of jobs
for job, path in jobs:
    print(f"Procesando y descargando en {path}", end="...") 
    collect_input_data.wait_and_download(job, path, poll_interval=60)
    print(f"Descargado") 


print(f"Todos las imágenes {var} procesadas, puedes continuar")

Procesando BIOPAR LAI desde 2015-10-01 hasta 2025-09-30
para la extensión [-1.673, 38.438, -1.668, 38.442], esto puede tardar un momento
Se encontraron datos de Sentinel-2 en caché para el mes 10 de 2015. Omitiendo descarga.
Se encontraron datos de Sentinel-2 en caché para el mes 11 de 2015. Omitiendo descarga.
Se encontraron datos de Sentinel-2 en caché para el mes 12 de 2015. Omitiendo descarga.
Se encontraron datos de Sentinel-2 en caché para el mes 01 de 2016. Omitiendo descarga.
Se encontraron datos de Sentinel-2 en caché para el mes 02 de 2016. Omitiendo descarga.
Se encontraron datos de Sentinel-2 en caché para el mes 03 de 2016. Omitiendo descarga.
Se encontraron datos de Sentinel-2 en caché para el mes 04 de 2016. Omitiendo descarga.
Se encontraron datos de Sentinel-2 en caché para el mes 05 de 2016. Omitiendo descarga.
Se encontraron datos de Sentinel-2 en caché para el mes 06 de 2016. Omitiendo descarga.
Se encontraron datos de Sentinel-2 en caché para el mes 07 de 2016. Omi

2026-04-27 14:47:40,620 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: created
2026-04-27 14:48:41,380 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: running
2026-04-27 14:49:42,167 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: running
2026-04-27 14:50:42,290 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: running
2026-04-27 14:51:42,621 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: running
2026-04-27 14:52:42,980 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: running
2026-04-27 14:53:43,249 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: running
2026-04-27 14:54:43,540 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: running
2026-04-27 14:55:43,881 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: running
2026-04-27 14:56:44,058 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: running
2026-04-27 14:57:44,205 [INFO] Job j-2604271246324635ac08e42ff51a2379 status: running
2026-04-27 14:58:44,360 [INFO] Job j-2604271246324635a

Descargado
Procesando y descargando en mystorage/301a-biophysical/agramon/input/s2_202409_LAI.tif...

2026-04-27 15:04:55,691 [INFO] Job j-260427124701447eb9b7a9a035d47087 status: finished
2026-04-27 15:04:57,144 [INFO] Downloading Job result asset 'openEO.tif' from https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-260427124701447eb9b7a9a035d47087/openEO.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=91a233ef8acb4619a9277bee6d0b7054%2F20260427%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20260427T130456Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYwNDI3MTI0NzAxNDQ3ZWI5YjdhOWEwMzVkNDcwODciXSwidXNlcl9pZCI6WyJkNjlhMzc5My1lMzUzLTRlY2MtOThhYS04OTQ5OGYyY2Q2NjAiXX0sInRyYW5zaXRpdmVfdGFnX2tleXM

Descargado
Procesando y descargando en mystorage/301a-biophysical/agramon/input/s2_202507_LAI.tif...

2026-04-27 15:04:57,988 [INFO] Job j-26042712472548f6b7b26e425d35fc86 status: finished
2026-04-27 15:04:59,797 [INFO] Downloading Job result asset 'openEO.tif' from https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-26042712472548f6b7b26e425d35fc86/openEO.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=d729a52a7df84e32b0c43e8e40230f4e%2F20260427%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20260427T130458Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYwNDI3MTI0NzI1NDhmNmI3YjI2ZTQyNWQzNWZjODYiXSwidXNlcl9pZCI6WyJkNjlhMzc5My1lMzUzLTRlY2MtOThhYS04OTQ5OGYyY2Q2NjAiXX0sInRyYW5zaXRpdmVfdGFnX2tleXM

Descargado
Todos las imágenes LAI procesadas, puedes continuar


:::{attention} Atención
Si has recibido algún error tipo `ConcurrentJobLimit: Job was not started because concurrent job limit (30) is reached` ejecuta la celda [](#remove-jobs) para eliminar todos los trabajos pendientes de la nube y reintentar
:::

### Descargar promedios mensuales de Contenido de Clorofila en el dosel
Haz lo mismo para el producto CCC

In [ ]:
MAX_JOBS = 24
        
date_ini = dt.datetime(w_years.value[0], 10, 1)
date_end = dt.datetime(w_years.value[1], 9, 30)
bbox = bboxs[0]
bbox_polygon = eval(to_geojson(box(*bbox)))
var = "CCC"
input_dir = aoi_data_dir / "input"

if not input_dir.exists():
    input_dir.mkdir()

time_window = [str(date_ini.date()), str(date_end.date())]
print(f"Procesando BIOPAR {var} desde {time_window[0]} hasta {time_window[1]}\n"
      f"para la extensión {bbox}, esto puede tardar un momento")

bio = biophysical_processing.get_biopar(
    connection, var, time_window, bbox_polygon
    )

date = date_ini
jobs = []
count = 0
while date < date_end:    
    dates_range = [str(date.date()),  
                   str((date + relativedelta(months=1) - dt.timedelta(days=1)).date())]    
    bio_month = bio.filter_temporal(dates_range).reduce_dimension(dimension="t", reducer="mean")  

    s2_path = input_dir / f"s2_{date:%Y%m}_{var}.tif"
    if not s2_path.exists():
        print(f"Creando trabajo para el {var} mensual de Sentinel-2 desde {dates_range[0]} hasta {dates_range[1]}")
        job = bio_month.create_job(out_format="GTiff")
        job.start()
        jobs.append([job, s2_path])
    else:
        print(f"Se encontraron datos de Sentinel-2 en caché para el mes {date:%m} de {date:%Y}. Omitiendo descarga.")  
    
    date += relativedelta(months=1)
    count += 1
    # Download data every MAX_DOWNLOADS
    if count > MAX_JOBS:
        for job, path in jobs:
            print(f"Procesando y descargando en {path}", end="...") 
            collect_input_data.wait_and_download(job, path, poll_interval=60)
            print(f"Descargado") 
        # Restart job list and counter
        count = 0        
        jobs = []

# Process the last batch of jobs
for job, path in jobs:
    print(f"Procesando y descargando en {path}", end="...") 
    collect_input_data.wait_and_download(job, path, poll_interval=60)
    print(f"Descargado") 


print(f"Todos las imágenes {var} procesadas, puedes continuar")

Procesando BIOPAR CCC desde 2015-10-01 hasta 2025-09-30
para la extensión [-1.673, 38.438, -1.668, 38.442], esto puede tardar un momento
Creando trabajo para el CCC mensual de Sentinel-2 desde 2015-10-01 hasta 2015-10-31
Creando trabajo para el CCC mensual de Sentinel-2 desde 2015-11-01 hasta 2015-11-30
Creando trabajo para el CCC mensual de Sentinel-2 desde 2015-12-01 hasta 2015-12-31
Creando trabajo para el CCC mensual de Sentinel-2 desde 2016-01-01 hasta 2016-01-31
Creando trabajo para el CCC mensual de Sentinel-2 desde 2016-02-01 hasta 2016-02-29
Creando trabajo para el CCC mensual de Sentinel-2 desde 2016-03-01 hasta 2016-03-31
Creando trabajo para el CCC mensual de Sentinel-2 desde 2016-04-01 hasta 2016-04-30
Creando trabajo para el CCC mensual de Sentinel-2 desde 2016-05-01 hasta 2016-05-31
Creando trabajo para el CCC mensual de Sentinel-2 desde 2016-06-01 hasta 2016-06-30
Creando trabajo para el CCC mensual de Sentinel-2 desde 2016-07-01 hasta 2016-07-31
Creando trabajo para el

2026-04-27 16:03:32,532 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running


Procesando y descargando en mystorage/301a-biophysical/agramon/input/s2_201510_CCC.tif...

2026-04-27 16:04:32,653 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:05:32,784 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:06:32,985 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:07:33,202 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:08:33,592 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:09:33,738 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:10:33,893 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:11:34,116 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:12:34,678 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:13:34,848 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:14:35,240 [INFO] Job j-2604271356014b73ba6f06af2979a857 status: running
2026-04-27 16:15:35,390 [INFO] Job j-2604271356014b73b

Descargado
Procesando y descargando en mystorage/301a-biophysical/agramon/input/s2_201511_CCC.tif...

...

2026-04-27 16:49:14,140 [INFO] Job j-260427135819458589368f32d38a0260 status: finished
2026-04-27 16:49:17,193 [INFO] Downloading Job result asset 'openEO.tif' from https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-260427135819458589368f32d38a0260/openEO.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=de090f8efd1840d49f9d18cfe6e72ed6%2F20260427%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20260427T144914Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYwNDI3MTM1ODE5NDU4NTg5MzY4ZjMyZDM4YTAyNjAiXSwidXNlcl9pZCI6WyJkNjlhMzc5My1lMzUzLTRlY2MtOThhYS04OTQ5OGYyY2Q2NjAiXX0sInRyYW5zaXRpdmVfdGFnX2tleXM

Descargado
Procesando y descargando en mystorage/301a-biophysical/agramon/input/s2_201607_CCC.tif...

2026-04-27 16:49:19,000 [INFO] Job j-26042713583742228dea1cd5edc5ba8c status: finished
2026-04-27 16:49:21,702 [INFO] Downloading Job result asset 'openEO.tif' from https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-26042713583742228dea1cd5edc5ba8c/openEO.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=a9437639bf5b4e6080e4f2ebacf131ff%2F20260427%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20260427T144919Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYwNDI3MTM1ODM3NDIyMjhkZWExY2Q1ZWRjNWJhOGMiXSwidXNlcl9pZCI6WyJkNjlhMzc5My1lMzUzLTRlY2MtOThhYS04OTQ5OGYyY2Q2NjAiXX0sInRyYW5zaXRpdmVfdGFnX2tleXM

Descargado
Procesando y descargando en mystorage/301a-biophysical/agramon/input/s2_201608_CCC.tif...

2026-04-27 16:50:22,561 [INFO] Job j-260427135853400491144fb25de84ee9 status: running
2026-04-27 16:51:22,699 [INFO] Job j-260427135853400491144fb25de84ee9 status: running


:::{attention} Atención
Si has recibido algún error tipo `ConcurrentJobLimit: Job was not started because concurrent job limit (30) is reached` ejecuta la celda [](#remove-jobs) para eliminar todos los trabajos pendientes de la nube y reintentar
:::

## De CCC a clorofila foliar
Del contenido de clorofila en el dosel `CCC` y el Índice de área foliar `LAI` podemos derivar la clorifila foliar:

:::{math}
C_{a+b} \left(\mu\textrm{g m}^{-2}\right)= \frac{CCC \left(\mu\textrm{g m}^{-2}\right)}{LAI}
:::


In [ ]:
def read_raster(path):
    src = rasterio.open(path)
    profile = src.meta
    values = src.read(1)
    src.close()
    return values, profile 
    
lai_images = sorted(list(input_dir.glob(f"s2_*_LAI.tif")))
for lai_path in range(lai_images):
    date = lai_path.stem.split("_")[1]
    ccc_path = input_dir / f"s2_{date}_CCC.tif"
    cab_path = input_dir / f"s2_{date}_CAB.tif"
    if cab_path.exists():
        print(f"{cab_path} ya generado, omitiendo")
    else:
        lai, profile = read_raster(lai_path)
        ccc, _ = read_raster(ccc_path)
        cab = ccc / lai
        save_raster(cab_path, cab, profile)

# Extraer tendencias y estacionalidad

## Selecciona una capa
Por defecto, las tendencias se van a calcular para el promedio de todos los píxeles (según la extensión que dibujaste anteriormente)

En esta celda puedes en cambio subir una capa geojson con los polígonos sobre los que quieres hacer los cálculos. De este modo se sacarían las tendencias promedio

In [ ]:
w_file = widgets.FileUpload(
    value = (),
    accept='.geojson',
    multiple=False
)
display(w_file)

### Selecciona variable a procesar

In [ ]:
w_var = widgets.Dropdown(
    options=["LAI", "CAB", "CCC"],
    value='LAI',
    description='Variable:',
    tooltip="Selecciona rasgo biofísico a procesar")
display(w_var)

In [ ]:
if len(w_shape.value) == 0:
    file = aoi_data_dir / f"{aoi_name}.geojson"
else:
    uploaded_file = w_shape.value[0]
    ext = uploaded_file.name.split(".")[-1]
    print(ext)
    file = aoi_data_dir / f"{aoi_name}.{ext}"
    with open(file, "wb") as fp:
        fp.write(uploaded_file.content.tobytes())

site_data = gpd.read_file(file).explode()

dates = []
df = {"date":[], "fid": [], "value": []}
images = sorted(list(input_dir.glob(f"s2_*_{var}.tif")))
if len(w_file.value) == 0:
    for image in images:
        print(f"Calculando el promedio de {image}")
        date = dt.datetime.strptime(image.stem.split("_")[1], "%Y%m%")
        src = rasterio.open(image)
        src.close()
        data = np.nanmean(src.read())
        df["date"].append(date)
        df["value"].append(data)
        df["fid"].append(0)

else:
    for image in images:
        print(f"Calculando el promedio zonal de {image}")
        stats = zonal_stats(
            w_file.value["name"], image, stats="mean", geojson_out=True, all_touched=True)
        date = dt.datetime.strptime(image.stem.split("_")[1], "%Y%m%")
        df["date"].append(date)
        for stat in stats.:            
            df["value"].append(stat['properties']["mean"])
            df["fid"].append(stat['properties']["fid"])
            
df = pd.DataFrame(df)
print(df.head)

## Descomposición estacional de las extracciones

In [ ]:
stl_kwargs = {"seasonal_deg": 0,
              "trend_deg": 0}

out_dir = data_dir / "output"
if not out_dir.is_dir():
    out_dir.mkdir(parents=True)
    
out_file = out_dir / f"zonal_trends_{var}.csv"
ts_dict = {"fid": [], "date": [], "values", "trend": []}

fig = go.Figure()
for i in data["fid"].unique():
    id = data["fid"] == i
    subset = data.loc[id]
    subset = subset.set_index("date")

    ts = MSTL(subset,
              periods=12, windows=5*12+1, iterate=5,
              stl_kwargs=stl_kwargs).fit()
    ts_array.append(ts)
    ax2.plot(ts.trend.index, ts.trend.values, label=i)
    ts_dict["fid"].append(i)
    ts_dict["date"] += ts.trend.index.to_list()
    ts_dict["values"] += ts.observed.values.to_list()
    ts_dict["trend"] += ts.trend.values.to_list()
    ts_dict["fid"] += np.full_like(ts.observed.values, fid).to_list()
    fig.add_trace(go.Scatter(x=ts.trend.index, y=ts.trend.values, name=f"site-{i}", mode="lines"))

ts_dict = pd.DataFrame(ts_dict)
ts_dict.to_csv(out_file, sep=";")
print(f"Guardadas las tendencias en {out_file}")
fig.update_layout(title_text=f"Tendencia anual para {var}", xaxis_title="Fecha", yaxis_title=var)

(remove-jobs)=
# Eliminar trabajos actuales
Si has recibido algún error tipo `ConcurrentJobLimit: Job was not started because concurrent job limit (30) is reached` ejecuta esta celda para eliminar todos los trabajos pendientes de la nube y reintentar

In [11]:
jobs = connection.list_jobs()
for job in jobs:
    job = connection.job(job["id"])
    print(f"Borrando trabajo {job}")
    job.delete()

print("Todos los trabajos pendientes en cola eliminados, puedes volver a procesar los productos")

<BatchJob job_id='j-260426155722407cb6efc01dd2a8ea8c'>
<BatchJob job_id='j-26042615570643769ef95bdb8cf5cc0f'>
<BatchJob job_id='j-26042615564548509f1f8820ba567abc'>
<BatchJob job_id='j-26042615562348faa00515a9285cc3f8'>
<BatchJob job_id='j-2604261555554e29b8c42af971452733'>
<BatchJob job_id='j-2604261555384493a9871b7df00c92f0'>
<BatchJob job_id='j-2604261555224eb593950d1f17ca412d'>
<BatchJob job_id='j-2604261548554581992ec4791b491cc3'>
<BatchJob job_id='j-2604261548394977aedaa564683d592e'>
<BatchJob job_id='j-260426153623448e892680780305b60e'>
<BatchJob job_id='j-26042615360845338bb69566030ba9d4'>
<BatchJob job_id='j-26042615352945eea41b11cdd3291453'>
<BatchJob job_id='j-2604261535084e3dbbb3c58e4bcbcea4'>
<BatchJob job_id='j-260426153449457692f412f0693262ec'>
<BatchJob job_id='j-2604261534304898ac98c735d10048eb'>
<BatchJob job_id='j-260426153407486ca895b2eded849543'>
<BatchJob job_id='j-26042615335142b9ad37962cc3e80e8c'>
<BatchJob job_id='j-26042615333641c089fdad9bd9fe5e62'>
<BatchJob 